# 03 - Dynamics And Control

## What / Why / How

**What we are trying to do:** Connect commands to motion using dynamics, PID control, actuator limits, and differential-drive movement.

**Why this matters:** Real robots have inertia, delay, saturation, and wheel geometry. Controllers must handle those physical limits.

**How we will do it:** Simulate a controlled mass, compare soft versus aggressive gains, then drive a differential robot through turns and toward a waypoint.

## Goal

Learn how actions change motion:

- Dynamics: forces and torques cause acceleration.
- Control: feedback chooses actions based on error.
- Saturation: real motors have limits.
- Differential drive: many mobile robots move by left and right wheel speeds.

In [ ]:
import math
import random
from collections import defaultdict

import numpy as np

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see the plot: pip install -r requirements.txt")

## PID Control On A Simple Mass

In [ ]:
def simulate_mass_pid(kp=8.0, kd=3.0, ki=0.0, force_limit=8.0, dt=0.02, steps=250):
    x, v = -1.0, 0.0
    target = 1.0
    integral = 0.0
    rows = []
    for step in range(steps):
        error = target - x
        integral += error * dt
        force = kp * error - kd * v + ki * integral
        force = np.clip(force, -force_limit, force_limit)
        acceleration = force  # mass = 1 kg
        v += acceleration * dt
        x += v * dt
        rows.append((step * dt, x, v, error, force))
    return np.array(rows)

runs = {
    "soft": simulate_mass_pid(kp=3, kd=1.5),
    "balanced": simulate_mass_pid(kp=8, kd=3),
    "aggressive": simulate_mass_pid(kp=18, kd=1),
}
for name, rows in runs.items():
    print(name, "final x", round(float(rows[-1, 1]), 3), "max force", round(float(np.max(np.abs(rows[:, 4]))), 3))

In [ ]:
if HAS_PLOT:
    plt.figure(figsize=(8, 3))
    for name, rows in runs.items():
        plt.plot(rows[:, 0], rows[:, 1], label=name)
    plt.axhline(1.0, color="black", linestyle="--")
    plt.xlabel("time [s]")
    plt.ylabel("position")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()
else:
    plot_unavailable()

## Differential Drive Robot

In [ ]:
def diff_drive_step(pose, left_speed, right_speed, wheel_base=0.4, dt=0.05):
    x, y, theta = pose
    v = 0.5 * (left_speed + right_speed)
    omega = (right_speed - left_speed) / wheel_base
    x += v * np.cos(theta) * dt
    y += v * np.sin(theta) * dt
    theta += omega * dt
    theta = np.arctan2(np.sin(theta), np.cos(theta))
    return np.array([x, y, theta])

pose = np.array([0.0, 0.0, 0.0])
path = []
for _ in range(120):
    pose = diff_drive_step(pose, left_speed=0.7, right_speed=1.0)
    path.append(pose.copy())
path = np.array(path)
print("final pose [x, y, heading]:", path[-1])

In [ ]:
if HAS_PLOT:
    plt.figure(figsize=(4, 4))
    plt.plot(path[:, 0], path[:, 1])
    plt.axis("equal")
    plt.grid(True, alpha=0.3)
    plt.title("Differential-drive arc")
    plt.show()
else:
    plot_unavailable()

## Waypoint Controller

In [ ]:
def drive_to_waypoint(start_pose, goal, steps=300):
    pose = np.array(start_pose, dtype=float)
    rows = []
    for step in range(steps):
        dx, dy = goal[0] - pose[0], goal[1] - pose[1]
        distance = np.hypot(dx, dy)
        desired_heading = np.arctan2(dy, dx)
        heading_error = np.arctan2(np.sin(desired_heading - pose[2]), np.cos(desired_heading - pose[2]))
        v = np.clip(1.0 * distance, 0.0, 0.8)
        omega = np.clip(3.0 * heading_error, -2.0, 2.0)
        left = v - 0.2 * omega
        right = v + 0.2 * omega
        pose = diff_drive_step(pose, left, right)
        rows.append(pose.copy())
        if distance < 0.03:
            break
    return np.array(rows)

wp_path = drive_to_waypoint([0, 0, 0], goal=np.array([2.0, 1.2]))
print("steps:", len(wp_path), "final:", wp_path[-1])

## Exercises

1. Make the PID aggressive enough to overshoot.
2. Add actuator saturation and compare trajectories.
3. Change the wheel base. How does turning change?
4. Add a second waypoint and drive a small route.